<a href="https://colab.research.google.com/github/mattkellymt/blks/blob/main/cuda_101.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CUDA 101: The CUDA Kernel

This introduction to CUDA programming can be run in Google Colab to provide free access to an NVIDIA Tesla T4 GPU.

In [ ]:
%%bash
rm -rf /content/*
nvidia-smi

Sat Oct 10 01:12:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
%%writefile parse_num.hpp
#pragma once
#include <charconv>
#include <cstdlib>
#include <string_view>
#include <system_error>
#include <tuple>

template <typename T>
std::tuple<T, int> parse_num(const std::string_view arg) {
  auto arg_end = arg.data() + arg.size();
  T temp_val = 0;
  auto [read_end, ec] = std::from_chars(arg.data(), arg_end, temp_val);
  bool parse_error = ec != std::errc{};
  bool partial_read = read_end != arg_end;
  if (parse_error || partial_read) {
    return {T{}, EXIT_FAILURE};
  }
  return {temp_val, EXIT_SUCCESS};
}

Writing parse_num.hpp


In [ ]:
%%writefile set_grid_shape.cuh
#pragma once
#include <cstdlib>
#include <cstdint>
#include <string_view>
#include <utility>
#include <cuda_runtime.h>
#include "parse_num.hpp"

inline int check_grid_shape(uint32_t x, uint32_t y, uint32_t z, const cudaDeviceProp& prop) {
  if (x == 0 || y == 0 || z == 0) {
    return EXIT_FAILURE;
  }
  if (x > prop.maxGridSize[0] || y > prop.maxGridSize[1] || z > prop.maxGridSize[2]) {
    return EXIT_FAILURE;
  }
  return EXIT_SUCCESS;
}

inline std::pair<dim3, int> set_grid_shape(uint32_t x, uint32_t y, uint32_t z, const cudaDeviceProp& prop) {
  if (check_grid_shape(x, y, z, prop) != EXIT_SUCCESS) {
    return {dim3{}, EXIT_FAILURE};
  }
  return {dim3(x, y, z), EXIT_SUCCESS};
}

inline std::pair<dim3, int> set_grid_shape(std::string_view xstr, std::string_view ystr, std::string_view zstr, const cudaDeviceProp& prop) {
  auto [x, err1] = parse_num<uint32_t>(xstr);
  auto [y, err2] = parse_num<uint32_t>(ystr);
  auto [z, err3] = parse_num<uint32_t>(zstr);
  if (err1 != EXIT_SUCCESS || err2 != EXIT_SUCCESS || err3 != EXIT_SUCCESS) {
    return {dim3{}, EXIT_FAILURE};
  }
  return set_grid_shape(x, y, z, prop);
}

Writing set_grid_shape.cuh


In [ ]:
%%writefile set_block_shape.cuh
#pragma once
#include <cstdlib>
#include <cstdint>
#include <string_view>
#include <tuple>
#include <cuda_runtime.h>
#include "parse_num.hpp"

inline int check_block_shape(uint32_t x, uint32_t y, uint32_t z, const cudaDeviceProp& prop) {
  if (x == 0 || y == 0 || z == 0) {
    return EXIT_FAILURE;
  }
  auto threads_per_block = x * y * z;
  if (threads_per_block > prop.maxThreadsPerBlock) {
    return EXIT_FAILURE;
  }
  return EXIT_SUCCESS;
}

inline std::tuple<dim3, int> set_block_shape(uint32_t x, uint32_t y, uint32_t z, const cudaDeviceProp& prop) {
  if (check_block_shape(x, y, z, prop) != EXIT_SUCCESS) {
    return {dim3{}, EXIT_FAILURE};
  }
  return {dim3(x, y, z), EXIT_SUCCESS};
}

inline std::tuple<dim3, int> set_block_shape(std::string_view xstr, std::string_view ystr, std::string_view zstr, const cudaDeviceProp& prop) {
  auto [x, err1] = parse_num<uint32_t>(xstr);
  auto [y, err2] = parse_num<uint32_t>(ystr);
  auto [z, err3] = parse_num<uint32_t>(zstr);
  if (err1 != EXIT_SUCCESS || err2 != EXIT_SUCCESS || err3 != EXIT_SUCCESS) {
    return {dim3{}, EXIT_FAILURE};
  }
  return set_block_shape(x, y, z, prop);
}

Writing set_block_shape.cuh


In [ ]:
%%writefile get_prop.cuh
#pragma once
#include <cstdlib>
#include <cstdint>
#include <tuple>
#include <cuda_runtime.h>

inline std::tuple<cudaDeviceProp, int> get_prop(uint32_t device_id) {
  cudaDeviceProp prop;
  cudaError_t err = cudaGetDeviceProperties(&prop, device_id);
  if (err != cudaSuccess) {
    return {cudaDeviceProp{}, EXIT_FAILURE};
  }
  return {prop, EXIT_SUCCESS};
}

Writing get_prop.cuh


In [ ]:
%%writefile threads_homogeneous.cuh
#pragma once
#include <cstdio>

__host__ __device__
void print(const char* msg) {
  printf("%s\n", msg);
}

Writing threads_homogeneous.cuh


In [ ]:
%%writefile threads_host.cuh
#pragma once
#include <string>
#include "threads_homogeneous.cuh"

__host__
void welcome() {
  std::string msg = "This program will launch a GPU kernel and print its thread number";
  print(msg.c_str());
}

Writing threads_host.cuh


In [ ]:
%%writefile threads_device.cuh
#pragma once
#include <cstdint>

__device__
uint32_t get_thread() {
  return blockIdx.x * blockDim.x + threadIdx.x;
}

Writing threads_device.cuh


In [ ]:
%%writefile threads_kernel.cuh
#pragma once
#include <cstdio>
#include "threads_device.cuh"
#include "threads_homogeneous.cuh"

__global__
void kernel() {
  uint32_t thread = get_thread();
  printf("Hello from thread %u!\n", thread);
}

Writing threads_kernel.cuh


In [ ]:
%%writefile threads.cu
#include <cstdlib>
#include <format>
#include <iostream>
#include <span>
#include <string>
#include <utility>
#include "get_prop.cuh"
#include "set_block_shape.cuh"
#include "set_grid_shape.cuh"
#include "threads_host.cuh"
#include "threads_kernel.cuh"

int main(int argc, char* argv[]) {
  std::span<char*> args(argv, argc);
  constexpr uint32_t device_id = 0;

  if (args.size() != 3) {
    std::string msg = std::format("Usage: {} <grid size> <block size>", args[0]);
    std::cerr << msg << std::endl;
    std::exit(EXIT_FAILURE);
  }

  auto err1 = cudaSetDevice(device_id);
  if (err1 != cudaSuccess) {
    std::cerr << "Error setting device" << std::endl;
    std::exit(EXIT_FAILURE);
  }

  auto [prop, err2] = get_prop(device_id);
  if (err2 != EXIT_SUCCESS) {
    std::cerr << "Error getting device properties" << std::endl;
    std::exit(EXIT_FAILURE);
  }

  auto [grid_shape, err3] = set_grid_shape(args[1], "1", "1", prop);
  if (err3 != EXIT_SUCCESS) {
    std::cerr << "Invalid grid shape setup" << std::endl;
    std::exit(EXIT_FAILURE);
  }

  auto [block_shape, err4] = set_block_shape(args[2], "1", "1", prop);
  if (err4 != EXIT_SUCCESS) {
    std::cerr << "Invalid block shape setup" << std::endl;
    std::exit(EXIT_FAILURE);
  }

  welcome();
  kernel<<<grid_shape, block_shape>>>();

  auto err5 = cudaGetLastError();
  if (err5 != cudaSuccess) {
    std::string msg = std::format("Kernel launch failed: {}", cudaGetErrorString(err5));
    std::cerr << msg << std::endl;
    std::exit(EXIT_FAILURE);
  }

  auto err6 = cudaDeviceSynchronize();
  if (err6 != cudaSuccess) {
    std::string msg = std::format("cudaDeviceSynchronize failed: {}", cudaGetErrorString(err6));
    std::cerr << msg << std::endl;
    std::exit(EXIT_FAILURE);
  }

  return EXIT_SUCCESS;
}


Writing threads.cu


In [ ]:
%%bash
nvcc -std=c++20 -arch=native -O3 "threads.cu" -o threads

In [ ]:
%%bash
./threads 2 3

This program will launch a GPU kernel and print its thread number
Hello from thread 3!
Hello from thread 4!
Hello from thread 5!
Hello from thread 0!
Hello from thread 1!
Hello from thread 2!


# CUDA 102: Extending Python with Ctypes

In [ ]:
%%writefile lib_cuda.cuh
#pragma once
#include "parse_num.hpp"

struct pair {
  union {
    uint32_t u32;
    int32_t i32;
    float f32;
  };
  int ec;
};

extern "C" {
  pair parse_u32(const char* str);
  pair parse_i32(const char* str);
  pair parse_f32(const char* str);
}

Writing lib_cuda.cuh


In [ ]:
%%writefile lib_cuda.cu
#include <cstdint>
#include <string_view>
#include "lib_cuda.cuh"

extern "C" {
  pair parse_u32(const char* str) {
    auto [val, ec] = parse_num<uint32_t>(std::string_view(str));
    return pair{.u32 = val, .ec = ec};
  }

  pair parse_i32(const char* str) {
    auto [val, ec] = parse_num<int32_t>(std::string_view(str));
    return pair{.i32 = val, .ec = ec};
  }

  pair parse_f32(const char* str) {
    auto [val, ec] = parse_num<float>(std::string_view(str));
    return pair{.f32 = val, .ec = ec};
  }
}

Writing lib_cuda.cu


In [ ]:
%%bash
nvcc -std=c++20 -arch=native -O3 --compiler-options '-fPIC' -shared lib_cuda.cu -o lib_cuda.so

In [ ]:
%%writefile mod_cuda.py
import ctypes
from ctypes import c_char_p, c_float, c_int32, c_uint32, Structure, Union

class Result(Union):
    _fields_ = [
        ("u32", c_uint32),
        ("i32", c_int32),
        ("f32", c_float)
    ]

class Pair(Structure):
    _fields_ = [
        ("u", Result),
        ("ec", c_int32)
    ]

try:
    lib = ctypes.CDLL('./lib_cuda.so')
except OSError as e:
    raise RuntimeError("Failed to load lib_cuda.so") from e

lib.parse_u32.argtypes = [c_char_p]
lib.parse_u32.restype  = Pair

lib.parse_i32.argtypes  = [c_char_p]
lib.parse_i32.restype   = Pair

lib.parse_f32.argtypes  = [c_char_p]
lib.parse_f32.restype   = Pair

def parse_u32(s: str) -> int:
    res = lib.parse_u32(s.encode('utf-8'))
    if res.ec != 0:
        raise ValueError(f"Failed to parse '{s}' as uint32_t (error code: {res.ec})")
    return res.u.u32

def parse_i32(s: str) -> int:
    res = lib.parse_i32(s.encode('utf-8'))
    if res.ec != 0:
        raise ValueError(f"Failed to parse '{s}' as int32_t (error code: {res.ec})")
    return res.u.i32

def parse_f32(s: str) -> float:
    res = lib.parse_f32(s.encode('utf-8'))
    if res.ec != 0:
        raise ValueError(f"Failed to parse '{s}' as float (error code: {res.ec})")
    return res.u.f32


Writing mod_cuda.py


In [ ]:
%%writefile parse_u32.py
import sys
from mod_cuda import parse_u32

if __name__ == '__main__':
    if len(sys.argv) != 2:
        print(f"Usage: {sys.argv[0]} <number>")
        sys.exit(1)
    print(parse_u32(sys.argv[1]))

Writing parse_u32.py


In [ ]:
%%writefile parse_i32.py
import sys
from mod_cuda import parse_i32

if __name__ == '__main__':
    if len(sys.argv) != 2:
        print(f"Usage: {sys.argv[0]} <number>")
        sys.exit(1)
    print(parse_i32(sys.argv[1]))

Writing parse_i32.py


In [ ]:
%%writefile parse_f32.py
import sys
from mod_cuda import parse_f32

if __name__ == '__main__':
    if len(sys.argv) != 2:
        print(f"Usage: {sys.argv[0]} <number>")
        sys.exit(1)
    print(parse_f32(sys.argv[1]))

Writing parse_f32.py


In [ ]:
%%bash
python3 parse_u32.py 42
python3 parse_i32.py -42
python3 parse_f32.py 3.14

42
-42
3.140000104904175


# CUDA 103: Extending Python with C

In [ ]:
%%writefile lib_cuda_module.cpp
#define PY_SSIZE_T_CLEAN
#include <Python.h>
#include <cstdint>
#include "lib_cuda.cuh"

static PyObject* py_parse_u32(PyObject* self, PyObject* args) {
    const char* str;
    if (!PyArg_ParseTuple(args, "s", &str)) {
        return nullptr;
    }
    pair res = parse_u32(str);
    if (res.ec != 0) {
        PyErr_SetString(PyExc_ValueError, "Failed to parse input as uint32_t");
        return nullptr;
    }
    return PyLong_FromUnsignedLong(res.u32);
}

static PyObject* py_parse_i32(PyObject* self, PyObject* args) {
    const char* str;
    if (!PyArg_ParseTuple(args, "s", &str)) {
        return nullptr;
    }
    pair res = parse_i32(str);
    if (res.ec != 0) {
        PyErr_SetString(PyExc_ValueError, "Failed to parse input as int32_t");
        return nullptr;
    }
    return PyLong_FromLong(res.i32);
}

static PyObject* py_parse_f32(PyObject* self, PyObject* args) {
    const char* str;
    if (!PyArg_ParseTuple(args, "s", &str)) {
        return nullptr;
    }
    pair res = parse_f32(str);
    if (res.ec != 0) {
        PyErr_SetString(PyExc_ValueError, "Failed to parse input as float");
        return nullptr;
    }
    return PyFloat_FromDouble(res.f32);
}

static PyMethodDef LibCudaMethods[] = {
    {"parse_u32", py_parse_u32, METH_VARARGS, ""},
    {"parse_i32",  py_parse_i32,  METH_VARARGS, ""},
    {"parse_f32",  py_parse_f32,  METH_VARARGS, ""},
    {nullptr, nullptr, 0, nullptr}
};

static struct PyModuleDef libcudamodule = {
    PyModuleDef_HEAD_INIT,
    "direct_cuda",
    "",
    -1,
    LibCudaMethods
};

PyMODINIT_FUNC PyInit_direct_cuda(void) {
    return PyModule_Create(&libcudamodule);
}

Writing lib_cuda_module.cpp


In [ ]:
%%bash
PYTHON_CFLAGS=$(python3-config --cflags)
PYTHON_LDFLAGS=$(python3-config --ldflags --embed)
g++ -std=c++20 -O3 -fPIC -shared \
    ${PYTHON_CFLAGS} \
    lib_cuda_module.cpp \
    ./lib_cuda.so \
    -o direct_cuda.so \
    ${PYTHON_LDFLAGS}

In [ ]:
%%writefile direct_parse_u32.py
import sys
import direct_cuda

if __name__ == '__main__':
    if len(sys.argv) != 2:
        print(f"Usage: {sys.argv[0]} <number>")
        sys.exit(1)
    print(direct_cuda.parse_u32(sys.argv[1]))

Writing direct_parse_u32.py


In [ ]:
%%writefile direct_parse_i32.py
import sys
import direct_cuda

if __name__ == '__main__':
    if len(sys.argv) != 2:
        print(f"Usage: {sys.argv[0]} <number>")
        sys.exit(1)
    print(direct_cuda.parse_i32(sys.argv[1]))

Writing direct_parse_i32.py


In [ ]:
%%writefile direct_parse_f32.py
import sys
import direct_cuda

if __name__ == '__main__':
    if len(sys.argv) != 2:
        print(f"Usage: {sys.argv[0]} <number>")
        sys.exit(1)
    print(direct_cuda.parse_f32(sys.argv[1]))

Writing direct_parse_f32.py


In [ ]:
%%bash
python3 direct_parse_u32.py 42
python3 direct_parse_i32.py -42
python3 direct_parse_f32.py 3.14

42
-42
3.140000104904175
